# Programming for Data Science 
### Project Notebook: "Where should I live?" 
#### Group Members:
- Afonso Fernandes / 20241710
- Lourenço Lima / 20241711
- Pedro Jorge / 20241819
- David Morais / 20241759
## Project Repository
GitHub Repository:  
https://github.com/afonsolince06/-Where-should-I-live-PDS-Project

## **1. Introduction**
This notebook represents the final integration phase of our project, **"Where Should I Live?"**. Building upon the **Data Cleaning** (Part 1) and **Web Scraping** (Part 2) phases, we have developed a comprehensive tool designed to help users identify the <u>best European cities to live in</u> based on their personal preferences and constraints.

In this stage, we merge our cleaned datasets—containing metrics on <u>salary, cost of living, weather and demographics — with geospatial data</u> to create a unified analytical dataset.

### **2. Objectives**
The primary objective of this notebook is to provide an interactive decision-support system. Instead of static analysis, we empower the user to explore the data dynamically through a **Dashboard**.

Key goals include:
* **Data Integration:** Merging socio-economic metrics with scraped geographical coordinates.
* **Interactive Filtering:** Allowing users to filter cities based on financial, climatic, and social preferences.
* **Visualization:** Plotting qualifying cities on an interactive map for geospatial context.

### 3. **Dashboard Features**
The tool allows users to refine their search using the following criteria:

* **Financial:** 
    * **Minimum Average Monthly Salary:** Filter cities that meet income expectations.
    * **Maximum Average Cost of Living:** Filter cities that fit within a specific budget.
    * **Maximum Unemployment Rate:** Avoid cities with high economic instability.
* **Climate:**
    * **Heat Stress:** Limit the maximum number of days with very strong heat stress to ensure climatic comfort.
* **Social:**
    * **Languages:** Select cities where specific languages (e.g., English, Portuguese, Spanish) are widely spoken.

### **Import essential libraries and define an alias for them**

In [1]:
import pandas as pd 
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import re

### **Importing the already cleaned dataset with coordinates**

In [2]:
full_data=pd.read_csv('city_data_with_coords.csv') 
#Load the data with both information from the first and second notebook, such as average monthly salary and the coordinates of each city
full_data.head()

,Country,City,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Main Spoken Languages,Average Monthly Salary,Average Rent Price,Average Cost of Living,Last Data Update,Individual Languages,Spendable Income,Latitude,Longitude
0,Austria,Salzburg,243.0,375489,250472.0,20.30,3.0,66689.0,0,German,3200,1100,2186,2023-11-03,['German'],1014,47°48′00″N,13°02′42″E
1,Austria,Vienna,310.0,2983513,2018818.0,20.10,10.2,55770.0,3,"German, English, Turkish, Serbian",2500,1050,2061,2024-06-15,"['German', 'English', 'Turkish', 'Serbian']",439,48°12′30″N,16°22′21″E
2,Belgium,Antwerp,928.0,1139663,723396.0,27.70,6.2,57595.0,3,"Dutch, French, Arabic",2609,900,1953,2024-08-09,"['Dutch', 'French', 'Arabic']",656,51°13′04″N,04°24′01″E
3,Belgium,Bruges,840.0,119765,74600.0,25.63,5.6,63083.0,0,"Dutch, French",2700,1250,1708,2023-10-25,"['Dutch', 'French']",992,51°12′32″N,03°13′27″E
4,Belgium,Brussels,681.0,3284548,2137425.0,27.50,10.7,62500.0,3,"French, Dutch, Arabic, English",3350,1200,1900,2023-04-22,"['French', 'Dutch', 'Arabic', 'English']",1450,50°50′48″N,04°21′09″E


### **Creation of a List with every language that appears in the Dataset**

In [3]:
raw_langs = full_data['Main Spoken Languages'].astype(str).str.split(',')
flat_list = [lang.strip() for sublist in raw_langs if isinstance(sublist, list) for lang in sublist]# Flatten the list (turn list of lists into one big list) and strip whitespace
languages = sorted(list(set(flat_list))) #Get unique values for the list and sort them
print(f"Loaded {len(languages)} unique individual languages.")
print(languages)

Loaded 35 unique individual languages.
['Arabic', 'Bengali', 'Bulgarian', 'Catalan', 'Croatian', 'Czech', 'Danish', 'Dutch', 'English', 'Estonian', 'Finnish', 'French', 'German', 'Greek', 'Hungarian', 'Irish Gaelic', 'Italian', 'Latvian', 'Luxembourgish', 'Maltese', 'Norwegian', 'Polish', 'Portuguese', 'Romanian', 'Russian', 'Scots', 'Scots Gaelic', 'Serbian', 'Slovak', 'Slovene', 'Spanish', 'Swedish', 'Turkish', 'Urdu', 'Valencian']


### **Converting the Coordinates into the Decimal Format**
Creation of function that converts the coordinates from the DMS (Degrees, Minutes, Seconds) format that they were scraped in to a decimal format to be used in the map visualization.

In [4]:
def dms_to_decimal(dms_coords):
    # Check if the value is already a float/int (already converted)
    if isinstance(dms_coords, (float, int)):
        return dms_coords
        
    try:
        dms_parts = re.split('[°′″NEW]', str(dms_coords))
        degrees = float(dms_parts[0]) 
        minutes = float(dms_parts[1]) if len(dms_parts) > 1 and dms_parts[1] else 0 
        seconds = float(dms_parts[2]) if len(dms_parts) > 2 and dms_parts[2] else 0 

        decimal_coords = degrees + (minutes / 60) + (seconds / 3600) 

        if 'S' in dms_coords or 'W' in dms_coords:
            decimal_coords = -decimal_coords
            
        return decimal_coords
    except Exception as e:
        return None # Handle errors gracefully

# Apply conversion immediately to the full_data dataframe
full_data['Latitude'] = full_data['Latitude'].apply(dms_to_decimal)
full_data['Longitude'] = full_data['Longitude'].apply(dms_to_decimal)

### **Dashboard**
We developed an interactive dashboard that let users filter cities based on key criteria, including minimum ``Average Monthly Salary``, maximum ``Average Cost of Living``, ``Unemployment Rate``, ``Heat Stress`` frequency, and preferred ``Main Spoken Languages``. Upon applying these filters, the system dynamically updates to present a detailed dataset of qualifying cities alongside a map with their locations.

In [5]:
# SETUP the WIDGETS (Inputs)
min_salary_slider = widgets.IntSlider(
    value=2000, min=500, max=10000, step=100, 
    description='Min Salary (€):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

max_cost_slider = widgets.IntSlider(
    value=1500, min=500, max=5000, step=100, 
    description='Max Cost of Living (€):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

max_unemployer_slider = widgets.FloatSlider(
    value=5, min=0, max=25, step=0.5, 
    description='Max Unemployment Rate (%):', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)
max_heat_slider = widgets.IntSlider(
    value=10, min=0, max=60, step=1, 
    description='Max Days of Heat Stress:', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)
# Creation of checkbox list to the languages
# 1. Create a list of checkbox widgets (one for each language)
checkbox_objects = []
for lang in languages:
    checkbox_objects.append(widgets.Checkbox(value=False, description=lang, indent=False))

# 2. Put them inside a container (VBox)
# 3. Add Layout to make it scrollable so it doesn't take up the whole screen
checkbox_container = widgets.VBox(
    children=checkbox_objects, 
    layout=widgets.Layout(
        width='100%', 
        height='200px',       # Fixed height
        overflow_y='scroll',  # Adds a scrollbar
        border='1px solid black', 
        padding='5px'
    )
)

lang_label = widgets.Label("Select Languages (Scroll to see more):") # Label for the language box

run_button = widgets.Button(description="Find My City", button_style='success') # Button to trigger the update
output_area = widgets.Output()

# DEFINE THE INTERACTION
def on_button_click(b):
    with output_area:
        output_area.clear_output()
        
        # Get values from sliders
        user_min_salary = min_salary_slider.value
        user_max_cost = max_cost_slider.value
        user_max_unemployment= max_unemployer_slider.value
        user_max_heat_stress=max_heat_slider.value
        selected_langs = [box.description for box in checkbox_objects if box.value] # extract the selected boxes
        
        # Validation: If nothing is checked, warn the user 
        if not selected_langs:
            print("Please select at least one language from the list above.")
            return
        
        def check_lang_match(city_lang_str):
            if not isinstance(city_lang_str, str): return False
            # Split the city's languages into a list
            city_langs_list = [l.strip() for l in city_lang_str.split(',')]
            # Return True if there is an overlap (intersection)
            return any(lang in city_langs_list for lang in selected_langs)

        # Create a boolean mask using our check function
        language_mask = full_data['Main Spoken Languages'].apply(check_lang_match)
        
        # Filter the Master DataFrame (Use the one created in Step 2)
        results = full_data[
            (full_data['Average Monthly Salary'] >= user_min_salary) &
            (full_data['Average Cost of Living'] <= user_max_cost) &
            (full_data['Unemployment Rate'] <= user_max_unemployment) &
            (full_data['Days of very strong heat stress'] <= user_max_heat_stress) &
            (language_mask)
        ]
        
        if results.empty:
            print("No cities match your criteria! Try lowering your standards or asking for a raise 😉")
        else:
            print(f"Found {len(results)} cities matching your criteria:")
            display(results.tail(15)) # Show table with the last 15 cities selected
            
            # Create Map (Using the map from Advanced_Topic_Building_Iterative_Map notebook)
            fig_map = px.scatter_mapbox(
                results,
                lat="Latitude",
                lon="Longitude",
                hover_name="City",
                size="Population", # Optional
                color="Average Monthly Salary",
                color_continuous_scale="Turbo",
                zoom=2.7,
                size_max=25,
                height=700,
                mapbox_style="open-street-map",
                title="Recommended Cities"
            )
            fig_map.show()

            # Identify the cities that are best in each feature
            best_salary = results.loc[results['Average Monthly Salary'].idxmax()]
            best_cost = results.loc[results['Average Cost of Living'].idxmin()]
            best_unemployment = results.loc[results['Unemployment Rate'].idxmin()]
            best_heat = results.loc[results['Days of very strong heat stress'].idxmin()]
            
            # 2. Create a clean Summary DataFrame
            summary_data = [
                {
                    "Category": " Highest Monthly Salary", 
                    "City": best_salary['City'], 
                    "Value": f"{best_salary['Average Monthly Salary']:.0f} €"
                },
                {
                    "Category": "Lowest Cost of Living", 
                    "City": best_cost['City'], 
                    "Value": f"{best_cost['Average Cost of Living']:.0f} €"
                },
                {
                    "Category": "Lowest Unemployment Rate", 
                    "City": best_unemployment['City'], 
                    "Value": f"{best_unemployment['Unemployment Rate']}%"
                },
                {
                    "Category": "Best Climate (Least Heat Stress)", 
                    "City": best_heat['City'], 
                    "Value": f"{best_heat['Days of very strong heat stress']} days"
                }
            ]
            
            summary_df = pd.DataFrame(summary_data)
            
            # 3. Display the performance table
            print("\n" + "-"*40)
            print("TOP Cities AMONG SELECTED CITIES")
            print("-"*40)
            display(summary_df)

# DISPLAY EVERYTHING
run_button.on_click(on_button_click)
ui = widgets.VBox([
    min_salary_slider, 
    max_cost_slider,
    max_unemployer_slider,
    max_heat_slider,
    lang_label, 
    checkbox_container, 
    run_button
])

display(ui, output_area)

Output()

## **4. Final Conclusion:  "Where Should I Live?"**

This notebook marks the final stage of our project, successfully integrating raw data, automated web enrichment, and interactive analytics into a unified decision-support tool. By combining the rigorous cleaning from **Phase 1** with the geographical enrichment from **Phase 2**.

### **Project Synthesis**
Throughout this journey, we have achieved several key milestones:
1.  **From Raw Data to Insights:** We transformed a fragmented dataset into a structured framework, identifying that "nominal wealth" (Salary) is often misleading without considering the "real wealth" (**Spendable Income**).
2.  **Web Scraping:** The use of **Web Scraping (Selenium)** allowed us to overcome data limitations, providing the spatial context necessary to visualize the European urban landscape.
3.  **Dashboard:** Our final **Interactive Dashboard** empowers users to navigate complex trade-offs, such as choosing between a high-salary/high-heat city versus a more moderate, affordable alternative, based on their own priorities.

### **Final Reflections**
The "Where Should I Live?" tool proves that Data Science can simplify life-changing decisions.  By providing this transparency, we contribute to a more informed and mobile European citizenry.

This project demonstrates that behind every data point (be it a GPS coordinate or a GDP figure) lies a potential home, a career path, and a quality of life.